# FlowNP en glucosa (persona B) — todo en Colab (T4)

**Preparación (una vez):** sube `PROYECTO_colab.zip` a la raíz de tu Drive. Entorno de ejecución → Cambiar tipo → **T4 GPU**.

Todo se guarda en Drive (checkpoints en `flowNP/output/`, métricas en `glucose_np/results/FlowNP/`). Si la sesión expira, reconecta y vuelve a ejecutar desde arriba: cada celda reanuda o salta lo ya hecho.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/PROYECTO'   # <- ajusta la ruta si quieres otra
ZIP = '/content/drive/MyDrive/PROYECTO_colab.zip'
if not os.path.exists(f'{PROJ}/flowNP/glucose.py'):   # primera vez: descomprimir en Drive
    !unzip -q -o "{ZIP}" -d "{PROJ}"
%cd {PROJ}/flowNP
!nvidia-smi

In [ ]:
!pip -q install flow-matching==1.0.10 attrdictionary

## 1. Entrenamiento (semillas 0, 1, 2)
100k pasos por semilla; checkpoint cada 1.000 pasos y `best.tar` según la pérdida FM en `e1_val`. Si existe `ckpt.tar` reanuda; si ya terminó, no hace nada.

In [ ]:
import os
for seed in [0, 1, 2]:
    d = f'output/results/glucose/default/fnp/seed{seed}'
    resume = '--resume' if os.path.exists(f'{d}/ckpt.tar') else ''
    !python glucose.py --mode=train --seed={seed} --print_freq=1000 {resume}

Progreso (útil tras reconectar):

In [ ]:
!tail -n 3 output/results/glucose/default/fnp/seed*/train_*.log

## 2. E4: tiempo frente a pasos del solver (semilla 0)
Antes de la evaluación final: la curva CRPS-pasos decide `STEPS` de la celda 3.
C debe medir TNP en una T4 con 100 muestras y batch 16 (hipótesis 3).

In [ ]:
!python glucose.py --mode=e4 --seed=0 --e4_steps=1,2,5,10,20,50,100,200 --e4_methods=euler,midpoint

In [ ]:
import json
for m, rows in json.load(open('output/results/glucose/default/fnp/seed0/e4_all.json')).items():
    for r in rows: print(m, r['steps'], r['nfe'], f"{r['time_per_task_s']*1e3:.1f} ms", f"CRPS {r['crps_mgdl']:.2f}")

## 3. Evaluación completa (11 experimentos × 3 semillas)
Ajusta `STEPS` según E4. Salta experimentos ya guardados.

In [ ]:
STEPS = 50
for seed in [0, 1, 2]:
    !python glucose.py --mode=eval --seed={seed} --sample_method=midpoint --sample_steps={STEPS} --num_samples=100

## 4. Tablas y figuras (código de A)

In [ ]:
%cd {PROJ}/glucose_np
!pip -q install pandas scipy matplotlib
!python plots.py